# PDU predictive maintenance - breach-risk model

**Goal:** every 12 hours (00:00 and 12:00 UTC) score each PDU with the probability of a **sustained breach in the next 12 hours**, so technicians can do preventive work before the breach happens.

* **Label:** at least 5 minutes in the next 12 h whose minute-average voltage or temperature is outside the unit's *effective* breach limits (GLOBAL < MODEL < RACK < UNIT rules, taken from `pdu_latest_status`). Rows where the unit is already in a sustained breach in the 2 h before scoring time are excluded from training and evaluation - those are monitoring problems, not predictions.
* **Features:** Feature Views (Databricks Feature Store) rolling-window aggregations over `pdu_minute_stats` (1 h / 6 h / 24 h / 72 h) and the maintenance actions table, computed point-in-time correctly by `create_training_set`. Definitions live in `src/ml/pdu_pm_features.py`, shared with the batch scoring job.
* **Experiments:** 4 feature lists x 3 model types (+ a rule-based baseline), all tracked in MLflow.
* **Validation:** out-of-time. Train on the oldest period, select on a later validation period, and report final performance once on a separate, most-recent **holdout** period that is never used for selection.
* **Output:** the winning model registered in Unity Catalog with alias `champion`.

In [0]:
%pip install -q "databricks-feature-engineering>=0.16.0" lightgbm
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.text("schema", "pdu_monitoring")
dbutils.widgets.text("ops_history_schema", "ops_history")
dbutils.widgets.text("maintenance_table", "maintenance_actions")
dbutils.widgets.text("model_name", "pdu_breach_risk_model")
dbutils.widgets.text("experiment_path", "")
dbutils.widgets.text("holdout_days", "4")
dbutils.widgets.text("validation_days", "4")
dbutils.widgets.text("burn_in_hours", "72")
dbutils.widgets.text("top_k", "10")

import os, sys, json
sys.path.append(os.path.abspath("../ml"))
import pdu_pm_features as pm

cfg = pm.PMConfig(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    ops_schema=dbutils.widgets.get("ops_history_schema"),
    maintenance_table=dbutils.widgets.get("maintenance_table"),
)
MODEL_NAME = f"{cfg.catalog}.{cfg.schema}.{dbutils.widgets.get('model_name')}"
HOLDOUT_DAYS = int(dbutils.widgets.get("holdout_days"))
VALIDATION_DAYS = int(dbutils.widgets.get("validation_days"))
BURN_IN_HOURS = int(dbutils.widgets.get("burn_in_hours"))
TOP_K = int(dbutils.widgets.get("top_k"))

user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT = dbutils.widgets.get("experiment_path") or f"/Users/{user}/pdu_predictive_maintenance"
print(cfg, MODEL_NAME, EXPERIMENT, sep="\n")

In [0]:
from pyspark.sql import functions as F

spine = pm.label_spine(spark, cfg).cache()
summary = (spine.groupBy("scoring_ts")
           .agg(F.count("*").alias("units"), F.sum("label").alias("positives"),
                F.sum(F.col("already_breaching").cast("int")).alias("already_breaching"),
                F.first("label_complete").alias("label_complete"))
           .orderBy("scoring_ts"))
display(summary)
print(f"{spine.count():,} unit x scoring-time rows, "
      f"{spine.where('label_complete').agg(F.avg('label')).first()[0]:.1%} positive (complete rows)")

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Maintenance features are only meaningful once actions have been recorded in the app.
maint_rows = spark.table(cfg.maintenance).count() if spark.catalog.tableExists(cfg.maintenance) else 0
INCLUDE_MAINT = maint_rows > 0
print(f"maintenance actions recorded: {maint_rows} -> include maintenance features: {INCLUDE_MAINT}")

raw_names = pm.all_raw_feature_names(include_maintenance=INCLUDE_MAINT)
features = pm.get_or_create_features(fe, cfg, raw_names, include_maintenance=INCLUDE_MAINT)
print(f"{len(features)} Feature Views features:")
for f_ in features:
    print("  ", getattr(f_, "full_name", None) or getattr(f_, "name", f_))

In [0]:
import pandas as pd

labeled = spine.where("label_complete")
training_set = fe.create_training_set(df=labeled, features=features, label="label")
pdf = pm.add_derived_features(training_set.load_df().toPandas())
pdf["scoring_ts"] = pd.to_datetime(pdf["scoring_ts"], utc=True)
print(pdf.shape)
display(pdf.head(5))

In [0]:
import matplotlib.pyplot as plt

t_min, t_max = pdf["scoring_ts"].min(), pdf["scoring_ts"].max()
holdout_start = t_max - pd.Timedelta(days=HOLDOUT_DAYS) + pd.Timedelta(hours=12)
val_start = holdout_start - pd.Timedelta(days=VALIDATION_DAYS)
train_start = t_min + pd.Timedelta(hours=BURN_IN_HOURS)   # 72 h windows fully populated

def split_of(ts):
    if ts < train_start: return "burn_in"
    if ts < val_start: return "train"
    if ts < holdout_start: return "validation"
    return "holdout"

pdf["split"] = pdf["scoring_ts"].map(split_of)
# The 12 h label window of a row never crosses into the next split because cadence == horizon,
# so no embargo rows are needed; features may look back into earlier splits (that is not leakage).
model_df = pdf[(pdf["split"] != "burn_in") & (~pdf["already_breaching"])].copy()

split_stats = (model_df.groupby("split")
               .agg(rows=("label", "size"), positives=("label", "sum"), positive_rate=("label", "mean"),
                    first=("scoring_ts", "min"), last=("scoring_ts", "max"))
               .reindex(["train", "validation", "holdout"]))
display(split_stats.reset_index())
print(f"excluded already-breaching rows: {int(pdf['already_breaching'].sum())}")

rate = model_df.groupby("scoring_ts")["label"].mean()
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(rate.index, rate.values, marker="o", lw=1)
for s, e, c in [(train_start, val_start, "tab:blue"), (val_start, holdout_start, "tab:orange"),
                (holdout_start, t_max + pd.Timedelta(hours=12), "tab:green")]:
    ax.axvspan(s, e, alpha=0.12, color=c)
ax.set_title("Share of units with a sustained breach in the next 12 h (blue=train, orange=validation, green=holdout)")
ax.set_ylabel("positive rate"); plt.tight_layout(); plt.show()

In [0]:
import numpy as np, mlflow, warnings
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from lightgbm import LGBMClassifier
warnings.filterwarnings("ignore")

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)

MODELS = {
    "logistic_regression": lambda: make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000)),
    "random_forest": lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=400, min_samples_leaf=5, max_features="sqrt",
                               class_weight="balanced_subsample", n_jobs=-1, random_state=7)),
    "lightgbm": lambda: LGBMClassifier(
        n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=20,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
        is_unbalance=True, random_state=7, verbose=-1),
}
FEATURE_LISTS = [fl for fl in pm.RAW_BY_LIST if fl != "full" or INCLUDE_MAINT]


def precision_recall_at_k(df, score, k=TOP_K):
    """Operational metric: technicians visit the top-k units each 12 h shift."""
    d = df.assign(score=score)
    top = d.sort_values("score", ascending=False).groupby("scoring_ts").head(k)
    return top["label"].mean(), top["label"].sum() / max(d["label"].sum(), 1)


def evaluate(df, score, prefix):
    p_at_k, r_at_k = precision_recall_at_k(df, score)
    out = {f"{prefix}_pr_auc": average_precision_score(df["label"], score),
           f"{prefix}_roc_auc": roc_auc_score(df["label"], score),
           f"{prefix}_precision_at_{TOP_K}": p_at_k, f"{prefix}_recall_at_{TOP_K}": r_at_k}
    if np.all((score >= 0) & (score <= 1)):
        out[f"{prefix}_brier"] = brier_score_loss(df["label"], score)
    return out


train = model_df[model_df.split == "train"]
val = model_df[model_df.split == "validation"]
holdout = model_df[model_df.split == "holdout"]
results = []

with mlflow.start_run(run_name="pm_experiment_grid") as parent:
    mlflow.log_params({"horizon_hours": cfg.horizon_hours, "min_breach_minutes": cfg.min_breach_minutes,
                       "train_rows": len(train), "validation_rows": len(val), "holdout_rows": len(holdout),
                       "val_start": str(val_start), "holdout_start": str(holdout_start),
                       "include_maintenance": INCLUDE_MAINT})
    # Rule baseline: rank units by how close their 1 h average temperature is to the breach limit.
    base = evaluate(val, -val["temp_headroom_1h"].fillna(99).values, "val")
    with mlflow.start_run(run_name="baseline_temp_headroom", nested=True):
        mlflow.log_params({"feature_list": "rule", "model_type": "temp_headroom_rule"})
        mlflow.log_metrics(base)
    results.append({"feature_list": "rule", "model_type": "baseline", **base})

    for fl in FEATURE_LISTS:
        cols = pm.model_columns(fl)
        for mname, make in MODELS.items():
            with mlflow.start_run(run_name=f"{fl}__{mname}", nested=True) as run:
                model = make().fit(train[cols], train["label"])
                p_val = model.predict_proba(val[cols])[:, 1]
                metrics = evaluate(val, p_val, "val")
                mlflow.log_params({"feature_list": fl, "model_type": mname, "n_features": len(cols)})
                mlflow.log_dict({"columns": cols}, "feature_columns.json")
                mlflow.log_metrics(metrics)
                results.append({"feature_list": fl, "model_type": mname, "run_id": run.info.run_id, **metrics})
                print(f"{fl:17s} {mname:20s} val PR-AUC {metrics['val_pr_auc']:.3f}  "
                      f"P@{TOP_K} {metrics[f'val_precision_at_{TOP_K}']:.2f}")

res = pd.DataFrame(results)
display(res.sort_values("val_pr_auc", ascending=False))

In [0]:
grid = res[res.model_type != "baseline"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in [(axes[0], "val_pr_auc", "Validation PR-AUC"),
                          (axes[1], f"val_precision_at_{TOP_K}", f"Validation precision@{TOP_K} per shift")]:
    piv = grid.pivot(index="feature_list", columns="model_type", values=metric).reindex(FEATURE_LISTS)
    im = ax.imshow(piv.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=20)
    ax.set_yticks(range(piv.shape[0]), piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i, j]:.3f}", ha="center", va="center", color="w")
    base_v = res.loc[res.model_type == "baseline", metric].iloc[0]
    ax.set_title(f"{title} (rule baseline {base_v:.3f})")
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

In [0]:
from mlflow.models import infer_signature

best = grid.sort_values(["val_pr_auc", f"val_precision_at_{TOP_K}"], ascending=False).iloc[0]
BEST_FL, BEST_MODEL = best["feature_list"], best["model_type"]
BEST_COLS = pm.model_columns(BEST_FL)
print(f"Selected on validation: {BEST_FL} + {BEST_MODEL} (val PR-AUC {best['val_pr_auc']:.3f})")

# Holdout check of the model exactly as selected (trained on train only) ...
sel_model = MODELS[BEST_MODEL]().fit(train[BEST_COLS], train["label"])
p_hold_sel = sel_model.predict_proba(holdout[BEST_COLS])[:, 1]
# ... and of the deployable model, refit on train + validation with the same configuration.
train_val = pd.concat([train, val])
final_model = MODELS[BEST_MODEL]().fit(train_val[BEST_COLS], train_val["label"])
p_hold = final_model.predict_proba(holdout[BEST_COLS])[:, 1]
hold_metrics = {**evaluate(holdout, p_hold, "holdout"),
                **{k.replace("holdout", "holdout_trainonly"): v
                   for k, v in evaluate(holdout, p_hold_sel, "holdout").items()},
                **{k.replace("val", "holdout_baseline"): v
                   for k, v in evaluate(holdout, -holdout["temp_headroom_1h"].fillna(99).values, "val").items()}}

with mlflow.start_run(run_name=f"champion__{BEST_FL}__{BEST_MODEL}") as champ:
    mlflow.log_params({"feature_list": BEST_FL, "model_type": BEST_MODEL, "n_features": len(BEST_COLS),
                       "trained_on": "train+validation", "selection_run_id": best["run_id"],
                       "horizon_hours": cfg.horizon_hours, "min_breach_minutes": cfg.min_breach_minutes})
    mlflow.log_metrics({**hold_metrics, "val_pr_auc": best["val_pr_auc"]})
    mlflow.log_dict({"feature_list": BEST_FL, "columns": BEST_COLS,
                     "raw_features": [n for n in pm.RAW_BY_LIST[BEST_FL] if INCLUDE_MAINT or "repair" not in n]},
                    "feature_columns.json")
    sig = infer_signature(train_val[BEST_COLS], final_model.predict_proba(train_val[BEST_COLS])[:, 1])
    info = mlflow.sklearn.log_model(final_model, artifact_path="model", signature=sig,
                                    input_example=train_val[BEST_COLS].head(3),
                                    registered_model_name=MODEL_NAME)
CHAMPION_RUN_ID = champ.info.run_id
display(pd.DataFrame([hold_metrics]).T.rename(columns={0: "holdout"}))

In [0]:
from sklearn.metrics import precision_recall_curve
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for score, name in [(p_hold, f"{BEST_FL} + {BEST_MODEL}"),
                    (-holdout["temp_headroom_1h"].fillna(99).values, "rule baseline")]:
    pr, rc, _ = precision_recall_curve(holdout["label"], score)
    axes[0].plot(rc, pr, label=f"{name} (AP {average_precision_score(holdout['label'], score):.3f})")
axes[0].axhline(holdout["label"].mean(), ls=":", c="grey", label="prevalence")
axes[0].set(xlabel="recall", ylabel="precision", title="Holdout precision-recall"); axes[0].legend()

frac_pos, mean_pred = calibration_curve(holdout["label"], p_hold, n_bins=8, strategy="quantile")
axes[1].plot(mean_pred, frac_pos, marker="o"); axes[1].plot([0, 1], [0, 1], ls=":", c="grey")
axes[1].set(xlabel="predicted probability", ylabel="observed breach rate", title="Holdout calibration")

d = holdout.assign(score=p_hold)
per_shift = (d.sort_values("score", ascending=False).groupby("scoring_ts").head(TOP_K)
             .groupby("scoring_ts")["label"].sum())
total = d.groupby("scoring_ts")["label"].sum()
axes[2].bar(total.index, total.values, width=0.3, color="lightgrey", label="units that breached")
axes[2].bar(per_shift.index, per_shift.values, width=0.3, color="tab:green", label=f"caught in top {TOP_K}")
axes[2].set(title=f"Holdout: breaches caught by visiting the top {TOP_K} each shift"); axes[2].legend()
plt.setp(axes[2].get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

In [0]:
from sklearn.inspection import permutation_importance

# Computed on the validation period with the train-only model, so the holdout stays untouched.
pi = permutation_importance(sel_model, val[BEST_COLS], val["label"], scoring="average_precision",
                            n_repeats=10, random_state=7, n_jobs=-1)
imp = (pd.Series(pi.importances_mean, index=BEST_COLS).sort_values().tail(15))
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax, xerr=pd.Series(pi.importances_std, index=BEST_COLS)[imp.index])
ax.set_title(f"Permutation importance (drop in validation PR-AUC) - {BEST_FL} + {BEST_MODEL}")
plt.tight_layout(); plt.show()
with mlflow.start_run(run_id=CHAMPION_RUN_ID):
    mlflow.log_figure(fig, "permutation_importance.png")

In [0]:
# For the holdout units that breached, plot predicted risk at each 12 h scoring time (bars)
# against the hourly max temperature from minute stats (line).
d = holdout.assign(risk=p_hold)
examples = d[d.label == 1].sort_values("risk", ascending=False)["unit_id"].drop_duplicates().head(3).tolist()
fig, axes = plt.subplots(len(examples), 1, figsize=(12, 3 * len(examples)), sharex=True)
axes = np.atleast_1d(axes)
for ax, unit in zip(axes, examples):
    hourly = (spark.table(cfg.minute_stats).where(F.col("unit_id") == unit)
              .where(F.col("minute_start") >= F.lit(str(val_start)))
              .groupBy(F.date_trunc("hour", "minute_start").alias("h"))
              .agg(F.max("temperature_avg").alias("t")).orderBy("h").toPandas())
    ax.plot(pd.to_datetime(hourly["h"], utc=True), hourly["t"], c="tab:red", lw=1)
    ax.axhline(d.loc[d.unit_id == unit, "temperature_breach_max"].iloc[0], ls="--", c="tab:red", alpha=0.5)
    ax.set_ylabel("temp C")
    ax2 = ax.twinx()
    u_ = d[d.unit_id == unit]
    ax2.bar(u_["scoring_ts"], u_["risk"], width=0.2, alpha=0.5,
            color=np.where(u_["label"] == 1, "tab:orange", "tab:blue"))
    ax2.set_ylim(0, 1); ax2.set_ylabel("risk")
    ax.set_title(f"{unit}: risk at each scoring time (orange = breach followed within 12 h)")
plt.tight_layout(); plt.show()
with mlflow.start_run(run_id=CHAMPION_RUN_ID):
    mlflow.log_figure(fig, "lead_time_examples.png")

In [0]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'")
              if v.run_id == CHAMPION_RUN_ID)
client.set_registered_model_alias(MODEL_NAME, "champion", version)
client.update_model_version(MODEL_NAME, version, description=(
    f"{BEST_FL} + {BEST_MODEL}; P(sustained breach in next {cfg.horizon_hours} h). "
    f"Holdout PR-AUC {hold_metrics['holdout_pr_auc']:.3f}, "
    f"precision@{TOP_K} {hold_metrics[f'holdout_precision_at_{TOP_K}']:.2f}."))
print(f"{MODEL_NAME} v{version} -> alias 'champion'")

## Conclusions

_Filled in after the run: best feature list and model, why it won, holdout performance versus the rule baseline, and caveats (synthetic data, short history)._